# Lab 12: End-to-End Workshop Portal

**Audience:** Participant

## Purpose
Validate the full group-specific workflow through the shared Workshop Portal.

## Theory
The Portal combines everything: your mission, the data bundle, the shared model, retrieval, queueing and tracing. Your group is locked after the first question, and your answers are saved in your own workspace.


## How to run
Run the cells from top to bottom; they are safe to run again.

- `LAB_MODE=local` uses explicit mocks or local services. Steps that need a GPU, CAI, Impala or NFS are reported as `SKIPPED` with a reason.
- `LAB_MODE=cai` uses the shared services from your environment. Nothing is mocked and a missing prerequisite is a `FAIL`.
- Save results only in your group workspace. Never paste secrets into a notebook.


In [ ]:
import json, os, sys
from pathlib import Path


def find_repo_root(start):
    for candidate in [start, *start.parents]:
        if (candidate / 'pyproject.toml').is_file() and (candidate / 'src' / 'bni_training').is_dir():
            return candidate
    if os.getenv('BNI_REPO_ROOT'):
        return Path(os.environ['BNI_REPO_ROOT'])
    raise RuntimeError('Open this notebook inside the repository checkout, or set BNI_REPO_ROOT.')


ROOT = find_repo_root(Path.cwd().resolve())
sys.path[:0] = [str(ROOT / 'src'), str(ROOT)]
LAB_MODE = os.getenv('LAB_MODE', 'local').lower()
assert LAB_MODE in {'local', 'cai'}, 'LAB_MODE must be local or cai'
# A group workspace keeps its settings next to the notebooks: <workspace>/config/group.env
group_env = Path.cwd().parent / 'config' / 'group.env'
if group_env.is_file() and not os.getenv('GROUP_ENV_FILE'):
    os.environ['GROUP_ENV_FILE'] = str(group_env)
print({'lab_mode': LAB_MODE, 'python': sys.version.split()[0], 'group_env': 'found' if os.getenv('GROUP_ENV_FILE') else 'not found (local defaults)'})

In [ ]:
from bni_training.workshop.lab_context import LabContextError, resolve_lab_context

try:
    context = resolve_lab_context(LAB_MODE)
    print(context.describe())
    context.close()
except LabContextError as exc:
    print('Group configuration problem:', exc)
for name in ['GROUP_ID', 'DATASET_ID', 'MISSION_ID', 'GROUP_WORKSPACE', 'DATA_BACKEND']:
    print(name, 'is set' if os.getenv(name) else 'is not set (taken from group.env or the local default)')

In [ ]:
from bni_training.lab_registry import get_lab, load_script_module

lab = get_lab('12')
module = load_script_module(lab.script_path)
print(lab.id, lab.title, '-', 'TRAINER ONLY' if lab.trainer_only else 'participant lab')

## Run the lab
The script below is the canonical implementation. It prints one check per line and finishes with a `LAB_STATUS` line.


In [ ]:
exit_code = module.main(['--mode', LAB_MODE])
outcome = {0: 'PASS', 3: 'SKIPPED'}.get(exit_code, 'FAIL')
print({'exit_code': exit_code, 'outcome': outcome})
assert outcome != 'FAIL', 'The lab reported FAIL; read the checks above'

## Review your saved output


In [ ]:
try:
    context = resolve_lab_context(LAB_MODE)
    saved = sorted(p.name for p in (context.workspace / 'output').glob('*.json'))
    print('Saved in your group workspace:', saved)
    context.close()
except LabContextError as exc:
    print('No workspace to list:', exc)

## Expected outcome
Locally an in-process Portal answers your first sample question; in CAI the Portal is reachable and your saved answer is found.

Every check is `PASS` or `SKIPPED` (with a reason). A `FAIL` stops the notebook.


## Checkpoint
Explain which component ran, which settings it used, and what you would expect if its service were unavailable.


## Optional challenge
Change one safe setting, rerun, and compare the results without touching shared services.


## Cleanup
Stop only resources assigned to your group. Do not stop shared vLLM, Qdrant, Phoenix, LiteLLM or the Workshop Portal.